In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.preprocessing import StandardScaler
import warnings, os
warnings.filterwarnings('ignore')
os.makedirs('images', exist_ok=True)

# ══════════════════════════════════════════════════════════════════
#  PHASE 3 — ANOMALY DETECTION
# ══════════════════════════════════════════════════════════════════
print("\n" + "="*65)
print("  PHASE 3: ANOMALY DETECTION")
print("="*65)

# ── Rule-Based Baseline ───────────────────────────────────────────
feature_df['rule_anomaly'] = (
    (feature_df['z_score'] < -1.8) |
    (feature_df['pct_dev_from_fleet'] < -20) |
    (feature_df['day_change_pct'] < -30) |
    (feature_df['ratio_to_rollmean'] < 0.65)
).astype(int)
print(f"✅ Rule-based anomalies   : {feature_df['rule_anomaly'].sum()}")

# ── Isolation Forest (unsupervised ML) ───────────────────────────
model_features = [
    'daily_yield_kwh', 'rolling_mean_7', 'rolling_std_7',
    'z_score', 'pct_dev_from_fleet', 'day_change_pct',
    'ratio_to_rollmean', 'performance_ratio'
]
X_raw    = feature_df[model_features].fillna(0)
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

iso = IsolationForest(contamination=0.07, random_state=42, n_estimators=200)
feature_df['iso_pred']   = iso.fit_predict(X_scaled)
feature_df['ml_anomaly'] = (feature_df['iso_pred'] == -1).astype(int)
print(f"✅ ML (Isolation Forest)  : {feature_df['ml_anomaly'].sum()}")

# Combined flag: flagged by EITHER rule OR ML
feature_df['anomaly_flag'] = (
    (feature_df['rule_anomaly'] == 1) | (feature_df['ml_anomaly'] == 1)
).astype(int)
total_anoms = int(feature_df['anomaly_flag'].sum())
print(f"✅ Combined anomaly flags : {total_anoms} "
      f"({total_anoms / len(feature_df) * 100:.1f}% of records)")

# ── Feature Importance via Random Forest ─────────────────────────
rf = RandomForestClassifier(
    n_estimators=200, random_state=42, class_weight='balanced'
)
rf.fit(X_scaled, feature_df['rule_anomaly'])
feat_imp = pd.DataFrame({
    'feature':    model_features,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(feat_imp['feature'], feat_imp['importance'],
               color='#5C6BC0', edgecolor='white')
for bar, val in zip(bars, feat_imp['importance']):
    ax.text(val + 0.002, bar.get_y() + bar.get_height() / 2,
            f'{val:.3f}', va='center', fontsize=9)
ax.set_title('Feature Importance — Random Forest Classifier',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')
ax.grid(axis='x', alpha=0.4)
plt.tight_layout()
plt.savefig('images/chart_feature_importance.png', dpi=150, bbox_inches='tight')
plt.close()
feat_imp.to_csv('feature_importance.csv', index=False)
print("✅ Feature importance chart + CSV saved")

# ══════════════════════════════════════════════════════════════════
#  PHASE 4 — EXPLAINABLE AI
# ══════════════════════════════════════════════════════════════════
print("\n" + "="*65)
print("  PHASE 4: EXPLAINABLE AI")
print("="*65)

def explain_row(row):
    """Plain-English reason for every flagged row."""
    if row['anomaly_flag'] == 0:
        return 'Normal Operation'
    reasons = []
    if row['z_score'] < -1.8:
        reasons.append(f"Low vs own history (z={row['z_score']:.2f})")
    if row['pct_dev_from_fleet'] < -20:
        reasons.append(
            f"{abs(row['pct_dev_from_fleet']):.0f}% below fleet avg")
    if row['day_change_pct'] < -30:
        reasons.append(
            f"Sharp drop from yesterday ({row['day_change_pct']:.0f}%)")
    if row['ratio_to_rollmean'] < 0.65:
        reasons.append(
            f"Only {row['ratio_to_rollmean']*100:.0f}% of 7-day avg")
    if row['ml_anomaly'] == 1 and not reasons:
        reasons.append('ML flagged unusual multi-metric pattern')
    return ' | '.join(reasons)

feature_df['explanation'] = feature_df.apply(explain_row, axis=1)

print("\n📋 Anomaly count by unit:")
print(feature_df.groupby(['source', 'unit_id'])['anomaly_flag']
      .sum().sort_values(ascending=False).to_string())

# Save final output
feature_df.to_csv('final_anomaly_results.csv', index=False)
print(f"\n✅ Saved: final_anomaly_results.csv "
      f"({len(feature_df)} rows, {len(feature_df.columns)} columns)")

print(f"""
╔══════════════════════════════════════════════════════════╗
║               PIPELINE COMPLETE — SUMMARY               ║
╠══════════════════════════════════════════════════════════╣
║  Records processed   : {len(feature_df):>6}                          ║
║  Anomalies detected  : {total_anoms:>6}                          ║
║  Rule-based flags    : {int(feature_df['rule_anomaly'].sum()):>6}                          ║
║  ML flags            : {int(feature_df['ml_anomaly'].sum()):>6}                          ║
║  Units monitored     : {feature_df['unit_id'].nunique():>6}                          ║
║  Date range          : {str(feature_df['date'].min().date())} → {str(feature_df['date'].max().date())}    ║
╚══════════════════════════════════════════════════════════╝
""")